<a href="https://colab.research.google.com/github/mohatamegha/Gen-AI-Fundamentals/blob/main/Vector_store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
pip install langchain chromadb google-generativeai langchain-core langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.6 MB/s eta 0:00:00


## Google Embedding Models

Google offers several powerful embedding models, often accessible through the Gemini API or specific model endpoints. These models convert text (or other data types like images) into numerical vectors, capturing semantic meaning. These embeddings are crucial for tasks like:

*   **Semantic Search:** Finding documents or passages related to a query, even if they don't share keywords.
*   **Recommendation Systems:** Suggesting similar items based on user preferences or item descriptions.
*   **Clustering:** Grouping similar pieces of text together.
*   **Classification:** Categorizing text based on its content.

### Key Google Embedding Models:

1.  **`text-embedding-004` (via Gemini API):** This is a popular and versatile text embedding model available through the Google Generative AI API (often referred to as the Gemini API). It's designed for a wide range of tasks and provides high-quality embeddings for English text. It's generally a good default choice for many applications.

2.  **`universal-sentence-encoder` (USE):** While not exclusively a Gemini API model, USE is a widely used and well-regarded family of embedding models from Google, available in TensorFlow Hub. It's known for its ability to produce highly semantic embeddings for sentences and short paragraphs.

3.  **Specialized Models:** Google also develops more specialized embedding models for specific domains or modalities (e.g., image embeddings like those used in Vision models, or embeddings for code).

### How to use them with the `google-generativeai` library (Python):

You can typically access `text-embedding-004` like this (after setting up your API key):

```python
import google.generativeai as genai

# Assuming you have your API key configured
# genai.configure(api_key="YOUR_API_KEY")

model = genai.GenerativeModel('text-embedding-004')

# To get an embedding for a piece of text
response = model.embed_content(content=["This is a sample sentence to embed."])
embedding = response['embedding'][0]
print(embedding)
```

For more advanced use cases or different models, you might integrate with libraries like LangChain which can wrap these models, or directly use TensorFlow Hub for models like USE.

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter

In [13]:
from langchain_core.documents import Document

In [15]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [16]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [20]:
# To securely use your API key, especially in Colab, you can store it in Colab's 'Secrets' tab.
# Then, you can access it like this:
import os
from google.colab import userdata

# Set the API key as an environment variable
# It's recommended to store your API key in Colab Secrets and access it via userdata.get()
# For example, if you saved your key as 'GOOGLE_API_KEY':
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


In [27]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model='gemini-embedding-2'),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [28]:
vector_store.add_documents(docs)

['7f2ab74d-a57f-4f8f-be9a-d6f5954604cd',
 '05e365fe-3445-4cca-b9b8-9f75b68e19c6',
 '0e5e14d7-3677-4804-872e-9d3c2d9f5dfb',
 '74990171-efc3-44a3-84d0-b8430c902416',
 '913ac48e-1c1b-4315-947d-aad7992a8a03']

In [31]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['7f2ab74d-a57f-4f8f-be9a-d6f5954604cd',
  '05e365fe-3445-4cca-b9b8-9f75b68e19c6',
  '0e5e14d7-3677-4804-872e-9d3c2d9f5dfb',
  '74990171-efc3-44a3-84d0-b8430c902416',
  '913ac48e-1c1b-4315-947d-aad7992a8a03'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [34]:
vector_store.similarity_search(query="Who among these is a bowler?", k=2) #k is the closest documents you wanna get.

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [33]:
vector_store.similarity_search_with_score(query="Who among these is a bowler?", k=2) #smallest score, better

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6153162717819214),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6540776491165161)]

In [39]:
vector_store.similarity_search_with_score(query="", filter={"team":"Chennai Super Kinds"}) #filtering by meta data

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6848995685577393),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7344018220901489)]

In [40]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='7f2ab74d-a57f-4f8f-be9a-d6f5954604cd', document=updated_doc1)

In [41]:
vector_store.delete(ids=['7f2ab74d-a57f-4f8f-be9a-d6f5954604cd'])

In [42]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['05e365fe-3445-4cca-b9b8-9f75b68e19c6',
  '0e5e14d7-3677-4804-872e-9d3c2d9f5dfb',
  '74990171-efc3-44a3-84d0-b8430c902416',
  '913ac48e-1c1b-4315-947d-aad7992a8a03'],
 'embeddings': array([[ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]]),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one 